# Fine-tune Grounding DINO on DIOR-RSVG (satellite imagery, text-query -> box)

**Goal:** given `(image, text query)`, return object coordinates (no image generation) so an
orchestrator/agent can call this as a tool and a separate script draws the boxes.

**Model:** Grounding DINO Tiny (Swin-T + BERT), fine-tuned with **Open-GroundingDino**
(https://github.com/longzw1997/Open-GroundingDino) — the community-standard training codebase for
Grounding DINO (the official IDEA-Research repo ships inference only, not training).

**Data:** DIOR-RSVG (https://github.com/ZhanYang-nwpu/RSVG-pytorch) — ~17.4K images / ~38K
referring expressions across 20 categories, built on the DIOR remote-sensing detection dataset.
**License: CC BY-NC 4.0 — research / non-commercial use only.** If your orchestrator is part of a
commercial product, keep that in mind before shipping the fine-tuned weights.

**Known rough edges to expect (flagging up front, not hiding them):**
1. The CUDA op build in Setup is the step most likely to need troubleshooting — it depends on the
   exact CUDA/PyTorch versions on whatever Kaggle image you land on.
2. `config/cfg_odvg.py`'s exact variable names for `batch_size` / `epochs` / `lr` aren't hard-coded
   here — open the file after Setup and check/adjust them yourself for your GPU (see the note
   before the training cell).
3. The checkpoint filename `train_dist.sh` writes isn't asserted blindly — the eval cell lists the
   output directory so you can confirm the real filename before loading it.
4. The XML parsing below mirrors the *official* `data_loader.py` from the DIOR-RSVG authors
   line-for-line (positional indexing `member[2]`=bbox, `member[3]`=expression), so it should be
   reliable — but do glance at one real annotation file after downloading, just to confirm.

Run cells top to bottom. Kaggle: enable **Internet** and a **GPU** (T4x2 or P100) in the notebook's
Settings panel before starting.

In [ ]:
import torch, subprocess
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))


## 1. Setup — clone Open-GroundingDino, install deps, build the CUDA ops

If the CUDA-op build fails, it's almost always a CUDA/PyTorch version mismatch. Common fix: check
`nvcc --version` vs `torch.version.cuda` and, if they disagree, either install a matching `nvcc`
via `conda install -c nvidia cuda-nvcc=<version>` or pin `torch` to match the system CUDA before
re-running the build.

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/longzw1997/Open-GroundingDino.git
%cd Open-GroundingDino
!pip install -q -r requirements.txt
# Open-GroundingDino calls BertModel.get_head_mask, which transformers 5.x removed -- the
# Kaggle image now ships 5.x, so training dies building the model without this pin.
!pip install -q "transformers<5"
!pip install -q gdown pycocotools


In [ ]:
%cd /kaggle/working/Open-GroundingDino/models/GroundingDINO/ops
!python setup.py build install
!python test.py   # should print a bunch of "True" — confirms the compiled op matches the pure-pytorch fallback
%cd /kaggle/working/Open-GroundingDino


## 2. Download pretrained weights (Swin-T checkpoint + warm the BERT cache)

In [ ]:
%cd /kaggle/working/Open-GroundingDino
!mkdir -p weights
!wget -q -P weights https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
print("Downloaded:", __import__("os").path.getsize("weights/groundingdino_swint_ogc.pth"), "bytes")

# bert-base-uncased auto-downloads from the HF hub the first time the model is built (public, no
# token needed) -- warm the cache here so training doesn't stall on it mid-run.
from transformers import AutoTokenizer, AutoModel
AutoTokenizer.from_pretrained("bert-base-uncased")
AutoModel.from_pretrained("bert-base-uncased")
print("BERT cached.")


## 3. Download DIOR-RSVG

Canonical source: the DIOR-RSVG authors' Google Drive folder (linked from
https://github.com/ZhanYang-nwpu/RSVG-pytorch). `gdown --folder` pulls the whole folder; if it
stalls or hits Google's "too many downloads" warning, re-run the cell (gdown resumes) or grab the
zip manually and upload it as a Kaggle Dataset instead — either way you want the layout below.

Expected layout:
```
DIOR_RSVG/
  Annotations/   *.xml   (bbox + referring expression per object)
  JPEGImages/    *.jpg
  train.txt      (26991 object-level indices)
  val.txt        (3829)
  test.txt       (7500)
```

In [ ]:
%cd /kaggle/working
!gdown --folder "https://drive.google.com/drive/folders/1hTqtYsC6B-m4ED2ewx5oKuYZV13EoJp_" -O DIOR_RSVG
!echo "---"
!find DIOR_RSVG -maxdepth 2 | head -20

# gdown pulls the folder's contents as-is -- sometimes that's already-extracted files, sometimes
# it's zip archives (Annotations.zip / JPEGImages.zip) that still need unzipping. Either way, any
# zip found gets extracted then DELETED immediately -- keeping both the zip and its extracted
# contents on disk at once was blowing past Kaggle's working-directory quota on a live run.
import glob, os
for zip_path in glob.glob("DIOR_RSVG/*.zip"):
    print(f"Extracting and removing {zip_path} ...")
    !unzip -q -o {zip_path} -d DIOR_RSVG
    os.remove(zip_path)
!echo "--- after extraction ---"
!find DIOR_RSVG -maxdepth 2 | head -20
!du -sh DIOR_RSVG


## 4. Parse the XML annotations into ODVG grounding JSONL

Mirrors the official `data_loader.py` exactly: for each `<object>` in each XML file,
`member[0]`=category name, `member[2]`=`bndbox` (xmin,ymin,xmax,ymax), `member[3]`=the referring
expression. `train.txt` / `val.txt` / `test.txt` are indices into this flattened
(image, object)-pair list, walked in the same sorted-filename order the original loader uses.

One JSONL line per referring expression (not grouped by image) — this matches how DIOR-RSVG,
RSVG-HR, and OPT-RSVG are all trained/evaluated in the literature: one (image, query, box) triplet
per sample.

In [ ]:
import os, json, pickle
import xml.etree.ElementTree as ET
from PIL import Image

DIOR_ROOT = "/kaggle/working/DIOR_RSVG"
ANNO_DIR  = os.path.join(DIOR_ROOT, "Annotations")
IMG_DIR   = os.path.join(DIOR_ROOT, "JPEGImages")

def load_split_ids(split):
    with open(os.path.join(DIOR_ROOT, f"{split}.txt")) as f:
        return set(int(x.strip()) for x in f if x.strip())

def get_image_size(xml_root, image_path):
    w_el, h_el = xml_root.find("./size/width"), xml_root.find("./size/height")
    if w_el is not None and h_el is not None:
        return int(w_el.text), int(h_el.text)
    with Image.open(image_path) as im:
        return im.size  # (width, height)

def parse_all_objects():
    xml_files = sorted(
        os.path.join(dp, f) for dp, _, fs in os.walk(ANNO_DIR) for f in fs if f.endswith(".xml")
    )
    records, count = [], 0
    for xp in xml_files:
        root = ET.parse(xp).getroot()
        filename = root.find("./filename").text
        w, h = get_image_size(root, os.path.join(IMG_DIR, filename))
        for member in root.findall("object"):
            category = member[0].text
            x1, y1, x2, y2 = (float(member[2][0].text), float(member[2][1].text),
                               float(member[2][2].text), float(member[2][3].text))
            expression = member[3].text
            records.append(dict(index=count, filename=filename, category=category,
                                 bbox=[x1, y1, x2, y2], width=w, height=h, expression=expression))
            count += 1
    return records

records = parse_all_objects()
n_images = len({r["filename"] for r in records})
print(f"Parsed {len(records)} (image, expression) pairs across {n_images} images")
categories = sorted({r["category"] for r in records})
print(f"{len(categories)} categories:", categories)


In [ ]:
def to_odvg_line(rec):
    x1, y1, x2, y2 = rec["bbox"]
    return json.dumps({
        "filename": rec["filename"],
        "height": rec["height"],
        "width": rec["width"],
        "grounding": {
            "caption": rec["expression"],
            "regions": [{"bbox": [x1, y1, x2, y2], "phrase": rec["expression"]}],
        },
    })

train_ids, val_ids, test_ids = load_split_ids("train"), load_split_ids("val"), load_split_ids("test")

train_lines = [to_odvg_line(r) for r in records if r["index"] in train_ids]
val_lines   = [to_odvg_line(r) for r in records if r["index"] in val_ids]
test_records = [r for r in records if r["index"] in test_ids]

os.makedirs("/kaggle/working/data", exist_ok=True)
with open("/kaggle/working/data/dior_rsvg_train_grounding.jsonl", "w") as f:
    f.write("\n".join(train_lines))
with open("/kaggle/working/data/dior_rsvg_val_grounding.jsonl", "w") as f:
    f.write("\n".join(val_lines))
with open("/kaggle/working/data/dior_rsvg_test_records.pkl", "wb") as f:
    pickle.dump(test_records, f)

print(f"train={len(train_lines)} (paper: 26991)  val={len(val_lines)} (paper: 3829)  "
      f"test={len(test_records)} (paper: 7500)")


## 5. Build a small COCO-format val set (for Open-GroundingDino's built-in periodic eval)

Their training loop's periodic validation only supports COCO-format detection data (fixed
categories, not free-text queries) — see the README. This is **only a training-time sanity signal**
("is box quality trending the right way"), not the metric that actually matters for your use case.
The real grounding accuracy (Acc@0.5 / Acc@0.7 / mIoU on text queries) is computed separately in
Section 9, after training, on the untouched test split.

Capped to a few hundred images to keep the periodic eval fast during a "quick" run — bump
`MAX_VAL_IMAGES` up if you want a more thorough in-training signal.

In [ ]:
MAX_VAL_IMAGES = 500

val_records = [r for r in records if r["index"] in val_ids]
cat2id = {c: i for i, c in enumerate(categories)}

val_by_image = {}
for r in val_records:
    val_by_image.setdefault(r["filename"], []).append(r)
val_image_names = list(val_by_image.keys())[:MAX_VAL_IMAGES]

images, annotations, ann_id = [], [], 0
for img_id, fname in enumerate(val_image_names):
    objs = val_by_image[fname]
    images.append({"id": img_id, "file_name": fname, "height": objs[0]["height"], "width": objs[0]["width"]})
    for r in objs:
        x1, y1, x2, y2 = r["bbox"]
        annotations.append({
            "id": ann_id, "image_id": img_id, "category_id": cat2id[r["category"]],
            "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2 - x1) * (y2 - y1), "iscrowd": 0,
        })
        ann_id += 1

coco_val = {
    "images": images,
    "annotations": annotations,
    "categories": [{"id": i, "name": c} for c, i in cat2id.items()],
}
with open("/kaggle/working/data/dior_rsvg_val_coco.json", "w") as f:
    json.dump(coco_val, f)
print(f"COCO val: {len(images)} images, {len(annotations)} boxes, {len(categories)} categories")


## 6. Point Open-GroundingDino at the data

In [ ]:
dataset_cfg = {
    "train": [{
        "root": "/kaggle/working/DIOR_RSVG/JPEGImages/",
        "anno": "/kaggle/working/data/dior_rsvg_train_grounding.jsonl",
        "dataset_mode": "odvg",   # pure grounding data -> no label_map needed
    }],
    "val": [{
        "root": "/kaggle/working/DIOR_RSVG/JPEGImages/",
        "anno": "/kaggle/working/data/dior_rsvg_val_coco.json",
        "label_map": None,
        "dataset_mode": "coco",
    }],
}
with open("/kaggle/working/Open-GroundingDino/config/datasets_dior_rsvg.json", "w") as f:
    json.dump(dataset_cfg, f, indent=2)
print("Wrote config/datasets_dior_rsvg.json")


## 7. Patch `config/cfg_odvg.py`

Per the README, evaluating on a non-COCO custom set needs `use_coco_eval = False` plus a
`label_list` of your class names. Both are applied programmatically below (the second line is just
*appended* — these config files are executed top-to-bottom as plain Python, so a later assignment
safely overrides anything set earlier).

**Do this part by hand once, right after running the cell below:** open
`Open-GroundingDino/config/cfg_odvg.py` and sanity-check (adjust for a T4/P100 + "quick run"):
- `batch_size` — start small (e.g. 2-4 per GPU) and raise it only if you don't hit an OOM.
- `epochs` — 5-10 is a reasonable first pass; the published fine-tuning writeups on this model see
  best validation performance in that range, with over-detection creeping in past it.
- `lr` / `lr_backbone` — the repo's defaults are a reasonable starting point; only touch these if
  loss is clearly diverging or stuck.

In [ ]:
cfg_path = "/kaggle/working/Open-GroundingDino/config/cfg_odvg.py"
with open(cfg_path) as f:
    cfg_text = f.read()

if "use_coco_eval = True" in cfg_text:
    cfg_text = cfg_text.replace("use_coco_eval = True", "use_coco_eval = False")
    with open(cfg_path, "w") as f:
        f.write(cfg_text)
    print("Set use_coco_eval = False")
else:
    print("WARNING: 'use_coco_eval = True' not found verbatim in cfg_odvg.py — "
          "open the file and set use_coco_eval = False by hand.")

with open(cfg_path, "a") as f:
    f.write(f"\nlabel_list = {categories!r}\n")
print("Appended label_list with", len(categories), "categories")

print("\n---- now open", cfg_path, "and check batch_size / epochs / lr as noted above ----")


## 8. Train

`GPU_NUM` below is auto-detected (2 for a T4x2 session, 1 for a P100 session). This will run for a
while — keep an eye on the loss printout; per the note above, a "quick" run is more like 5-10
epochs than the repo's pretraining-scale defaults.

In [ ]:
%cd /kaggle/working/Open-GroundingDino
import torch
GPU_NUM = max(1, torch.cuda.device_count())
print("Training with", GPU_NUM, "GPU(s)")


In [ ]:
%cd /kaggle/working/Open-GroundingDino
!bash train_dist.sh {GPU_NUM} config/cfg_odvg.py config/datasets_dior_rsvg.json ./logs/dior_rsvg_run1


In [ ]:
# Confirm the actual checkpoint filename(s) written — the exact name can vary by repo version,
# don't assume it, just look:
!ls -la /kaggle/working/Open-GroundingDino/logs/dior_rsvg_run1


## 9. The metric that actually matters: grounding accuracy on the held-out test split

This is a custom eval (Open-GroundingDino's built-in loop only does COCO-style detection mAP,
not per-query grounding accuracy). Standard RSVG protocol: for each (image, expression) test pair,
run the model with that expression as the text prompt, take the top-scoring box, and check its IoU
against the ground-truth box. Reports Acc@0.5 / Acc@0.7 / mIoU, the same metrics used in the
DIOR-RSVG / VRSBench papers, so your numbers are directly comparable to published baselines.

Runs on a 1000-example subset first (fast); drop `limit=` for the full 7500-item test set once
you're happy with the setup.

In [ ]:
import sys, os, pickle
sys.path.insert(0, "/kaggle/working/Open-GroundingDino")
import torch
from groundingdino.util.inference import load_model, load_image, predict

def box_cxcywh_to_xyxy_abs(box_norm, w, h):
    cx, cy, bw, bh = box_norm
    return [(cx - bw/2) * w, (cy - bh/2) * h, (cx + bw/2) * w, (cy + bh/2) * h]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def evaluate(ckpt_path, cfg_path, test_records, image_root, box_th=0.25, text_th=0.25, limit=None):
    model = load_model(cfg_path, ckpt_path)
    ious = []
    subset = test_records[:limit] if limit else test_records
    for i, r in enumerate(subset):
        img_path = os.path.join(image_root, r["filename"])
        image_source, image = load_image(img_path)
        boxes, logits, phrases = predict(model=model, image=image, caption=r["expression"],
                                          box_threshold=box_th, text_threshold=text_th)
        if len(boxes) == 0:
            ious.append(0.0)
            continue
        best_idx = int(logits.argmax())
        pred_xyxy = box_cxcywh_to_xyxy_abs(boxes[best_idx].tolist(), r["width"], r["height"])
        ious.append(iou_xyxy(pred_xyxy, r["bbox"]))
        if i % 500 == 0:
            print(i, "/", len(subset))
    n = len(ious)
    acc5 = sum(x >= 0.5 for x in ious) / n
    acc7 = sum(x >= 0.7 for x in ious) / n
    miou = sum(ious) / n
    return {"n": n, "Acc@0.5": acc5, "Acc@0.7": acc7, "mIoU": miou}

with open("/kaggle/working/data/dior_rsvg_test_records.pkl", "rb") as f:
    test_records = pickle.load(f)

image_root = "/kaggle/working/DIOR_RSVG/JPEGImages"
cfg_path = "/kaggle/working/Open-GroundingDino/tools/GroundingDINO_SwinT_OGC.py"

print("Zero-shot baseline (pretrained, not fine-tuned) on a 1000-item subset:")
print(evaluate("/kaggle/working/Open-GroundingDino/weights/groundingdino_swint_ogc.pth",
                cfg_path, test_records, image_root, limit=1000))


In [ ]:
# Fill in the actual checkpoint filename from the `ls` output in Section 8 above.
FINETUNED_CKPT = "/kaggle/working/Open-GroundingDino/logs/dior_rsvg_run1/checkpoint_best_regular.pth"

print("Fine-tuned model on the same 1000-item subset:")
print(evaluate(FINETUNED_CKPT, cfg_path, test_records, image_root, limit=1000))


## 10. Export

Copies the fine-tuned checkpoint + the config it was trained with into `/kaggle/working/output_model/`
so it shows up under this notebook's **Output** tab for download. Grab both files — the local
inference script (`grounding_tool.py`, alongside this notebook) needs the checkpoint plus the same
`GroundingDINO_SwinT_OGC.py` config to load it.

In [ ]:
import shutil, os
os.makedirs("/kaggle/working/output_model", exist_ok=True)
shutil.copy(FINETUNED_CKPT, "/kaggle/working/output_model/dior_rsvg_finetuned.pth")
shutil.copy("/kaggle/working/Open-GroundingDino/tools/GroundingDINO_SwinT_OGC.py",
            "/kaggle/working/output_model/GroundingDINO_SwinT_OGC.py")
print("Exported to /kaggle/working/output_model/ — download both files from the notebook's Output tab.")


## Next

Download `dior_rsvg_finetuned.pth` + `GroundingDINO_SwinT_OGC.py` from this notebook's Output tab,
then use them locally with `grounding_tool.py` (paired with this notebook) to call
`f(image, query) -> [{phrase, bbox_xyxy, score}, ...]` and draw boxes.